# Récupérer une API vers pandas (cours)

Exemples à **exécuter**. L'exercice (réponse d'API de commandes) est tout en bas.

### Un JSON imbriqué (comme une réponse d'API) — le problème

In [1]:
import pandas as pd
donnees = [
    {"id": 1, "montant": 120.5, "client": {"nom": "Durand", "ville": "Rouen"}},
    {"id": 2, "montant": 80.0,  "client": {"nom": "Martin", "ville": "Paris"}},
]
print(pd.DataFrame(donnees))   # la colonne 'client' contient des dictionnaires -> inexploitable

   id  montant                               client
0   1    120.5  {'nom': 'Durand', 'ville': 'Rouen'}
1   2     80.0  {'nom': 'Martin', 'ville': 'Paris'}


### pd.json_normalize aplatit l'imbrication

In [2]:
print(pd.json_normalize(donnees))   # 'client' déplié en client.nom / client.ville

   id  montant client.nom client.ville
0   1    120.5     Durand        Rouen
1   2     80.0     Martin        Paris


### En vrai : depuis une API (pattern requests — à décommenter si tu as internet)

In [11]:
import requests
r = requests.get("https://jsonplaceholder.typicode.com/users")  # API de test, sans authentification
print(r.status_code)      # 200 = OK
donnees = r.json() # -> liste de dicts
pd.json_normalize(donnees)

200


,id,name,username,email,phone,website,address.street,address.suite,address.city,address.zipcode,address.geo.lat,address.geo.lng,company.name,company.catchPhrase,company.bs
0,1,Leanne Graham,Bret,Sincere@april.biz,1-770-736-8031 x56442,hildegard.org,Kulas Light,Apt. 556,Gwenborough,92998-3874,-37.3159,81.1496,Romaguera-Crona,Multi-layered client-server neural-net,harness real-time e-markets
1,2,Ervin Howell,Antonette,Shanna@melissa.tv,010-692-6593 x09125,anastasia.net,Victor Plains,Suite 879,Wisokyburgh,90566-7771,-43.9509,-34.4618,Deckow-Crist,Proactive didactic contingency,synergize scalable supply-chains
2,3,Clementine Bauch,Samantha,Nathan@yesenia.net,1-463-123-4447,ramiro.info,Douglas Extension,Suite 847,McKenziehaven,59590-4157,-68.6102,-47.0653,Romaguera-Jacobson,Face to face bifurcated interface,e-enable strategic applications
3,4,Patricia Lebsack,Karianne,Julianne.OConner@kory.org,493-170-9623 x156,kale.biz,Hoeger Mall,Apt. 692,South Elvis,53919-4257,29.4572,-164.2990,Robel-Corkery,Multi-tiered zero tolerance productivity,transition cutting-edge web services
4,5,Chelsey Dietrich,Kamren,Lucio_Hettinger@annie.ca,(254)954-1289,demarco.info,Skiles Walks,Suite 351,Roscoeview,33263,-31.8129,62.5342,Keebler LLC,User-centric fault-tolerant solution,revolutionize end-to-end systems
5,6,Mrs. Dennis Schulist,Leopoldo_Corkery,Karley_Dach@jasper.info,1-477-935-8478 x6430,ola.org,Norberto Crossing,Apt. 950,South Christy,23505-1337,-71.4197,71.7478,Considine-Lockman,Synchronised bottom-line interface,e-enable innovative applications
6,7,Kurtis Weissnat,Elwyn.Skiles,Telly.Hoeger@billy.biz,210.067.6132,elvis.io,Rex Trail,Suite 280,Howemouth,58804-1099,24.8918,21.8984,Johns Group,Configurable multimedia task-force,generate enterprise e-tailers
7,8,Nicholas Runolfsdottir V,Maxime_Nienow,Sherwood@rosamond.me,586.493.6943 x140,jacynthe.com,Ellsworth Summit,Suite 729,Aliyaview,45169,-14.3990,-120.7677,Abernathy Group,Implemented secondary concept,e-enable extensible e-tailers
8,9,Glenna Reichert,Delphine,Chaim_McDermott@dana.io,(775)976-6794 x41206,conrad.com,Dayna Park,Suite 449,Bartholomebury,76495-3109,24.6463,-168.8889,Yost and Sons,Switchable contextually-based project,aggregate real-time technologies
9,10,Clementina DuBuque,Moriah.Stanton,Rey.Padberg@karina.biz,024-648-3804,ambrose.net,Kattie Turnpike,Suite 198,Lebsackbury,31428-2261,-38.2386,57.2232,Hoeger LLC,Centralized empowering task-force,target end-to-end models


# Exercices

Donnée : `data/commandes_api.json` — une **réponse d'API simulée** : une liste de commandes, avec un bloc `client` **imbriqué** (`nom`, `ville`, `segment`). **À toi de choisir tes outils.**

1. Charge le JSON et transforme-le en DataFrame **exploitable** (les infos client doivent devenir de vraies colonnes, pas un dict dans une case). Combien de commandes, et quelles colonnes obtiens-tu ?
2. On ne garde que les commandes **encaissées** (`statut == "payée"`). Sur ce sous-ensemble : **CA par ville** du client, de la plus rentable à la moins rentable. *(aplatir → filtrer → agréger → trier)*
3. Quel **segment** de client génère le plus gros **CA payé** ?
4. Réflexion (pas de code) : pourquoi `pd.json_normalize` plutôt qu'un simple `pd.DataFrame` sur ce JSON ?

In [ ]:
# --- CELLULE FOURNIE : on charge la "réponse d'API" depuis le fichier ---
import json, pandas as pd
with open("data/commandes_api.json", encoding="utf-8") as f:
    reponse_api = json.load(f)          # en vrai : reponse_api = requests.get(url).json()
print(type(reponse_api), "| nb éléments:", len(reponse_api))
reponse_api[0]                          # aperçu du 1er élément (structure imbriquée)

<class 'list'> | nb éléments: 22


{'commande_id': 'CMD001',
 'montant': 77.18,
 'statut': 'annulée',
 'client': {'nom': 'Roux', 'ville': 'Rouen', 'segment': 'Particulier'}}

In [ ]:
## 1
df = pd.json_normalize(reponse_api, sep="_")
display(df)
# display(pd.DataFrame(reponse_api)) ### PAS BON, car on a des données imbriquées dans client

,commande_id,montant,statut,client_nom,client_ville,client_segment
0,CMD001,77.18,annulée,Roux,Rouen,Particulier
1,CMD002,449.51,annulée,Roux,Lille,Premium
2,CMD003,282.95,en attente,Faure,Lille,Particulier
3,CMD004,21.22,payée,Martin,Lyon,Particulier
4,CMD005,235.83,en attente,Diaz,Paris,Pro
5,CMD006,140.73,payée,Moreau,Nantes,Premium
6,CMD007,56.63,payée,Martin,Nantes,Premium
7,CMD008,160.22,en attente,Benali,Nantes,Pro
8,CMD009,31.87,payée,Diaz,Lille,Pro
9,CMD010,63.53,annulée,Roux,Lille,Particulier


In [28]:
## 2
df_encaisse = df[df["statut"] == "payée"]
df_encaisse_groupby = df_encaisse.groupby("client_ville")["montant"].sum().sort_values(ascending=False)
display(df_encaisse_groupby)

# Paris est la ville avec le plus gros CA

client_ville
Paris     687.91
Lille     494.19
Nantes    410.36
Lyon       21.22
Name: montant, dtype: float64

In [31]:
## 3
df_segment_groupby = df_encaisse.groupby("client_segment")["montant"].sum().sort_values(ascending=False)
display(df_segment_groupby)

# Le premium

client_segment
Premium        1291.96
Particulier     289.85
Pro              31.87
Name: montant, dtype: float64

In [ ]:
## 4 (réflexion en commentaire)
# Car on a une imbrication de données des clients :nom, ville et segment donc pour remmetre ça sous la forme d'un df classique on utilise json_normalize,
# Sinon on aurait un dictionnaire contenant la ville, le nom et le segment du client sur une seule colonne au lieu d'avoir 3 colonnes contenant chacune une donnée.